# 📊 Visualisation & Analyse Exploratoire — Prédiction du Rendement Agricole

## Objectif
Explorer visuellement les données pour :
- Comprendre la distribution du **rendement** (variable cible)
- Identifier les **corrélations** entre facteurs climatiques, intrants et rendements
- Formuler des **hypothèses** exploitables pour la modélisation

## Structure
1. Setup & Chargement des données
2. Variable cible — Rendement agricole
3. Évolution temporelle du rendement
4. Top/Flop pays et cultures
5. Impact de la température sur le rendement
6. Impact des précipitations sur le rendement
7. Impact des intrants : engrais & pesticides
8. Analyse du bilan nutritif des sols
9. Matrice de corrélation globale
10. Hypothèses & Conclusions

---
## 0. Setup & Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# ── Style global ────────────────────────────────────────
PALETTE   = 'Set2'
BG_COLOR  = '#f8f9fa'
ACCENT    = '#2d6a4f'

plt.rcParams.update({
    'figure.figsize':     (14, 6),
    'figure.facecolor':   BG_COLOR,
    'axes.facecolor':     BG_COLOR,
    'axes.grid':          True,
    'grid.alpha':         0.35,
    'font.size':          11,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
})
sns.set_style('whitegrid')
sns.set_palette(PALETTE)

CLEAN_DIR = 'data/cleaned'
print('Librairies chargées ✅')

In [ ]:
# ── Chargement des datasets ─────────────────────────────
print('📦 Chargement des datasets en cours...')

df_prod   = pd.read_csv(f'{CLEAN_DIR}/production_cultures.csv')
df_fert   = pd.read_csv(f'{CLEAN_DIR}/fertilizers_nutrient.csv')
df_temp   = pd.read_csv(f'{CLEAN_DIR}/mean_temperature.csv')
df_precip = pd.read_csv(f'{CLEAN_DIR}/precipitations.csv')
df_pest   = pd.read_csv(f'{CLEAN_DIR}/pesticides.csv')
df_sols   = pd.read_csv(f'{CLEAN_DIR}/bilan_nutritif_sols.csv')
df_terres = pd.read_csv(f'{CLEAN_DIR}/intrants_utilisation_terres.csv')

print(f'✅ production_cultures  : {df_prod.shape}')
print(f'✅ fertilizers_nutrient : {df_fert.shape}')
print(f'✅ mean_temperature     : {df_temp.shape}')
print(f'✅ precipitations       : {df_precip.shape}')
print(f'✅ pesticides           : {df_pest.shape}')
print(f'✅ bilan_nutritif_sols  : {df_sols.shape}')
print(f'✅ intrants_terres      : {df_terres.shape}')
print('\nTous les datasets sont chargés !')

In [ ]:
# ── Extraction du rendement (kg/ha) ────────────────────
# Le rendement est notre VARIABLE CIBLE pour la prédiction
df_rendement = df_prod[df_prod['Element'] == 'Rendement'].copy()

print(f'Lignes de rendement (kg/ha) : {len(df_rendement):,}')
print(f'Pays couverts : {df_rendement["Pays"].nunique()}')
print(f'Cultures couverts : {df_rendement["Produit"].nunique()}')
print(f'Période : {df_rendement["Annee"].min()} – {df_rendement["Annee"].max()}')
print()
df_rendement.head()

---
## 1. Variable cible — Distribution du Rendement

**Hypothèse :** La distribution du rendement est très asymétrique car quelques cultures très intensives (blé, maïs) tirent la moyenne vers le haut.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribution du Rendement Agricole (kg/ha)', fontsize=15, fontweight='bold', y=1.01)

# Distribution brute
axes[0].hist(df_rendement['Valeur'].dropna(), bins=80, color=ACCENT, alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Rendement (kg/ha)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution brute (toutes cultures confondues)')
axes[0].axvline(df_rendement['Valeur'].median(), color='tomato', lw=2, ls='--', label=f'Médiane : {df_rendement["Valeur"].median():,.0f} kg/ha')
axes[0].axvline(df_rendement['Valeur'].mean(), color='orange', lw=2, ls='--', label=f'Moyenne : {df_rendement["Valeur"].mean():,.0f} kg/ha')
axes[0].legend()

# Distribution log
log_vals = np.log1p(df_rendement['Valeur'].dropna())
axes[1].hist(log_vals, bins=80, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('log(Rendement + 1)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution log-transformée (plus normale → meilleure pour les modèles ML)')

plt.tight_layout()
plt.show()

print('📌 Conclusion : La distribution brute est fortement asymétrique à droite.')
print('   → Pour les modèles de régression, une transformation log sera recommandée.')

---
## 2. Évolution temporelle du Rendement

**Hypothèse :** Le rendement moyen mondial augmente au fil des décennies grâce à la sélection variétale et l'intensification agricole.

In [ ]:
# Top 8 cultures les plus représentées
top_cultures = df_rendement.groupby('Produit')['Valeur'].count().nlargest(8).index.tolist()

df_top = df_rendement[df_rendement['Produit'].isin(top_cultures)]
evol    = df_top.groupby(['Annee', 'Produit'])['Valeur'].median().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Évolution temporelle du Rendement', fontsize=15, fontweight='bold')

# Toutes cultures — rendement médian mondial
global_evol = df_rendement.groupby('Annee')['Valeur'].median()
axes[0].plot(global_evol.index, global_evol.values, color=ACCENT, lw=2.5, marker='o', markersize=3)
axes[0].fill_between(global_evol.index, global_evol.values, alpha=0.1, color=ACCENT)
axes[0].set_title('Rendement médian mondial (toutes cultures)')
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Rendement médian (kg/ha)')

# Par culture
colors = sns.color_palette(PALETTE, len(top_cultures))
for i, cult in enumerate(top_cultures):
    sub = evol[evol['Produit'] == cult]
    axes[1].plot(sub['Annee'], sub['Valeur'], lw=2, label=cult, color=colors[i])

axes[1].set_title('Rendement médian par culture (Top 8)')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Rendement médian (kg/ha)')
axes[1].legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

print('📌 Conclusion : Le rendement mondial augmente tendanciellement.')
print('   → L\'année sera un prédicteur important du modèle (tendance long-terme).')

---
## 3. Top / Flop pays & cultures

**Hypothèse :** Les pays développés (USA, France, Pays-Bas) affichent des rendements bien supérieurs aux pays en développement, révélant l'importance des intrants et des pratiques agricoles.

In [ ]:
# Filtre sur les données récentes (après 2010) pour plus de pertinence
recents = df_rendement[df_rendement['Annee'] >= 2010]

top_pays = recents.groupby('Pays')['Valeur'].median().nlargest(15)
bot_pays = recents.groupby('Pays')['Valeur'].median().nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Rendement médian par pays (2010–présent)', fontsize=15, fontweight='bold')

top_pays.sort_values().plot(kind='barh', ax=axes[0], color=ACCENT, edgecolor='white')
axes[0].set_title('Top 15 — Meilleurs pays')
axes[0].set_xlabel('Rendement médian (kg/ha)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

bot_pays.sort_values(ascending=False).plot(kind='barh', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Flop 15 — Pays les moins productifs')
axes[1].set_xlabel('Rendement médian (kg/ha)')

plt.tight_layout()
plt.show()

In [ ]:
# Rendement par culture (toutes périodes)
top_cult = df_rendement.groupby('Produit')['Valeur'].median().nlargest(20)

fig, ax = plt.subplots(figsize=(14, 7))
colors_bar = sns.color_palette('coolwarm_r', len(top_cult))
top_cult.sort_values().plot(kind='barh', ax=ax, color=colors_bar, edgecolor='white')
ax.set_title('Top 20 cultures — Rendement médian mondial (kg/ha)', fontsize=14, fontweight='bold')
ax.set_xlabel('Rendement médian (kg/ha)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print('📌 Conclusion : Énorme disparité entre cultures.')
print('   → Le modèle devra inclure le type de culture comme feature categorielle (one-hot encoding).')

---
## 4. Impact de la Température sur le Rendement

**Hypothèse :** Il existe une température optimale pour le rendement. Au-delà d'un certain seuil, la chaleur devient néfaste (stress thermique). La relation n'est donc pas linéaire mais en forme de cloche.

In [ ]:
# Standardiser les noms de pays pour la jointure
df_temp_agg   = df_temp.groupby(['Pays', 'Annee'])['Valeur'].mean().reset_index().rename(columns={'Valeur': 'Temperature_C'})
df_rend_agg   = df_rendement.groupby(['Pays', 'Annee'])['Valeur'].median().reset_index().rename(columns={'Valeur': 'Rendement'})

df_merged_temp = pd.merge(df_rend_agg, df_temp_agg, on=['Pays', 'Annee'], how='inner')
print(f'Lignes après jointure Rendement × Température : {len(df_merged_temp):,}')
df_merged_temp.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Température vs Rendement Agricole', fontsize=15, fontweight='bold')

# Scatter plot
sample = df_merged_temp.sample(min(5000, len(df_merged_temp)), random_state=42)
axes[0].scatter(sample['Temperature_C'], sample['Rendement'], alpha=0.2, s=15, color='steelblue')
# Courbe de tendance
z = np.polyfit(sample['Temperature_C'].dropna(), sample['Rendement'].dropna(), 2)
p = np.poly1d(z)
x_line = np.linspace(sample['Temperature_C'].min(), sample['Temperature_C'].max(), 200)
axes[0].plot(x_line, p(x_line), color='tomato', lw=2.5, label='Tendance polynomiale (degré 2)')
axes[0].set_xlabel('Température moyenne (°C)')
axes[0].set_ylabel('Rendement médian (kg/ha)')
axes[0].set_title('Scatter : Température vs Rendement')
axes[0].legend()

# Boîte à moustaches par tranche de température
df_merged_temp['Tranche_T'] = pd.cut(df_merged_temp['Temperature_C'],
                                      bins=[-20, 0, 5, 10, 15, 20, 25, 30, 50],
                                      labels=['<0°C', '0-5°C', '5-10°C', '10-15°C',
                                              '15-20°C', '20-25°C', '25-30°C', '>30°C'])
bp_data = [df_merged_temp[df_merged_temp['Tranche_T'] == t]['Rendement'].dropna().values
           for t in df_merged_temp['Tranche_T'].cat.categories]
axes[1].boxplot(bp_data, labels=df_merged_temp['Tranche_T'].cat.categories, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_xlabel('Tranche de température')
axes[1].set_ylabel('Rendement (kg/ha)')
axes[1].set_title('Distribution du rendement par tranche de température')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('📌 Hypothèse confirmée : Le rendement est maximal entre 10°C et 20°C.')
print('   → La relation Température–Rendement est non-linéaire (quadratique).')
print('   → Ajouter Temperature² comme feature dans le modèle.')

---
## 5. Impact des Précipitations sur le Rendement

**Hypothèse :** Un niveau de précipitations modéré favorise un bon rendement. Trop peu (sécheresse) ou trop (inondation) nuisent à la production.

In [ ]:
df_precip_agg = df_precip.groupby(['Pays', 'Annee'])['Valeur'].mean().reset_index().rename(columns={'Valeur': 'Precipitations_mm'})

df_merged_precip = pd.merge(df_rend_agg, df_precip_agg, on=['Pays', 'Annee'], how='inner')
print(f'Lignes après jointure Rendement × Précipitations : {len(df_merged_precip):,}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Précipitations vs Rendement Agricole', fontsize=15, fontweight='bold')

sample2 = df_merged_precip.sample(min(5000, len(df_merged_precip)), random_state=42)
axes[0].scatter(sample2['Precipitations_mm'], sample2['Rendement'], alpha=0.2, s=15, color=ACCENT)
# Tendance
try:
    s = sample2.dropna(subset=['Precipitations_mm', 'Rendement'])
    z = np.polyfit(s['Precipitations_mm'], s['Rendement'], 2)
    p = np.poly1d(z)
    x_line = np.linspace(s['Precipitations_mm'].min(), s['Precipitations_mm'].max(), 200)
    axes[0].plot(x_line, p(x_line), color='tomato', lw=2.5, label='Tendance (degré 2)')
    axes[0].legend()
except Exception:
    pass
axes[0].set_xlabel('Précipitations (mm)')
axes[0].set_ylabel('Rendement médian (kg/ha)')
axes[0].set_title('Scatter : Précipitations vs Rendement')

# Boîtes par tranches
df_merged_precip['Tranche_P'] = pd.cut(df_merged_precip['Precipitations_mm'],
                                        bins=[0, 250, 500, 750, 1000, 1500, 2000, 5000],
                                        labels=['0-250', '250-500', '500-750', '750-1000',
                                                '1000-1500', '1500-2000', '>2000'])
bp_data2 = [df_merged_precip[df_merged_precip['Tranche_P'] == t]['Rendement'].dropna().values
            for t in df_merged_precip['Tranche_P'].cat.categories]
axes[1].boxplot(bp_data2, labels=df_merged_precip['Tranche_P'].cat.categories, patch_artist=True,
                boxprops=dict(facecolor=ACCENT, alpha=0.5))
axes[1].set_xlabel('Tranche de précipitations (mm)')
axes[1].set_ylabel('Rendement (kg/ha)')
axes[1].set_title('Distribution du rendement par tranche de précipitations')

plt.tight_layout()
plt.show()

print('📌 Hypothèse : Zone optimale vers 500-1000 mm de précipitations annuelles.')
print('   → Précipitations² à envisager comme feature.')

---
## 6. Impact des Engrais sur le Rendement

**Hypothèse :** L'usage d'engrais (NPK) est fortement corrélé positivement avec le rendement, et constitue l'un des prédicteurs les plus puissants du modèle.

In [ ]:
df_fert_agg = df_fert.groupby(['Pays', 'Annee'])['Valeur'].sum().reset_index().rename(columns={'Valeur': 'Engrais_kgha'})

df_merged_fert = pd.merge(df_rend_agg, df_fert_agg, on=['Pays', 'Annee'], how='inner')
print(f'Lignes après jointure Rendement × Engrais : {len(df_merged_fert):,}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Engrais vs Rendement Agricole', fontsize=15, fontweight='bold')

sample3 = df_merged_fert.sample(min(5000, len(df_merged_fert)), random_state=42)

# Scatter
axes[0].scatter(sample3['Engrais_kgha'], sample3['Rendement'], alpha=0.3, s=15, color='darkorange')
try:
    s = sample3.dropna(subset=['Engrais_kgha', 'Rendement'])
    z = np.polyfit(s['Engrais_kgha'], s['Rendement'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, s['Engrais_kgha'].quantile(0.99), 200)
    axes[0].plot(x_line, p(x_line), color='tomato', lw=2.5, label='Tendance linéaire')
    axes[0].legend()
except Exception:
    pass
axes[0].set_xlabel('Usage engrais total (kg/ha)')
axes[0].set_ylabel('Rendement médian (kg/ha)')
axes[0].set_title('Scatter : Engrais vs Rendement')
axes[0].set_xlim(0, df_merged_fert['Engrais_kgha'].quantile(0.99))

# Par type d'engrais (NPK)
fert_by_type = df_fert.groupby('Produit')['Valeur'].mean().nlargest(10).sort_values()
colors_fert = sns.color_palette('Oranges', len(fert_by_type))
fert_by_type.plot(kind='barh', ax=axes[1], color=colors_fert, edgecolor='white')
axes[1].set_title('Utilisation moyenne par type d\'engrais (kg/ha)')
axes[1].set_xlabel('Usage moyen (kg/ha)')

plt.tight_layout()
plt.show()

print('📌 Hypothèse : Corrélation positive entre intrants et rendement.')
print('   → Feature clé pour le modèle : dose totale d\'engrais N+P+K par ha.')

---
## 7. Impact des Pesticides sur le Rendement

**Hypothèse :** Les pesticides réduisent les pertes de récolte, donc corrélation positive avec le rendement. Mais l'effet marginal diminue au-delà d'un certain seuil.

In [ ]:
# Filtre : pesticides totaux uniquement
df_pest_total = df_pest[df_pest['Produit'].str.lower().str.contains('total', na=False)].copy()
if df_pest_total.empty:
    df_pest_total = df_pest.copy()

df_pest_agg = df_pest_total.groupby(['Pays', 'Annee'])['Valeur'].sum().reset_index().rename(columns={'Valeur': 'Pesticides_kgha'})

df_merged_pest = pd.merge(df_rend_agg, df_pest_agg, on=['Pays', 'Annee'], how='inner')
print(f'Lignes après jointure Rendement × Pesticides : {len(df_merged_pest):,}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Pesticides vs Rendement Agricole', fontsize=15, fontweight='bold')

sample4 = df_merged_pest.sample(min(3000, len(df_merged_pest)), random_state=42)
axes[0].scatter(sample4['Pesticides_kgha'], sample4['Rendement'], alpha=0.3, s=15, color='mediumpurple')
axes[0].set_xlabel('Pesticides (kg/ha)')
axes[0].set_ylabel('Rendement médian (kg/ha)')
axes[0].set_title('Scatter : Pesticides vs Rendement')
axes[0].set_xlim(0, df_merged_pest['Pesticides_kgha'].quantile(0.99))

# Évolution temporelle de l'usage de pesticides
pest_time = df_pest_total.groupby('Annee')['Valeur'].mean()
axes[1].plot(pest_time.index, pest_time.values, color='mediumpurple', lw=2.5, marker='o', markersize=3)
axes[1].fill_between(pest_time.index, pest_time.values, alpha=0.15, color='mediumpurple')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Usage moyen (kg/ha)')
axes[1].set_title('Évolution de l\'usage des pesticides dans le temps')

plt.tight_layout()
plt.show()

# Composition des pesticides
fig, ax = plt.subplots(figsize=(12, 5))
pest_by_cat = df_pest.groupby('Produit')['Valeur'].mean().nlargest(8).sort_values()
colors_pest = sns.color_palette('Purples', len(pest_by_cat))
pest_by_cat.plot(kind='barh', ax=ax, color=colors_pest, edgecolor='white')
ax.set_title('Utilisation moyenne par catégorie de pesticides (kg/ha)', fontsize=13, fontweight='bold')
ax.set_xlabel('Usage moyen (kg/ha)')
plt.tight_layout()
plt.show()

---
## 8. Bilan Nutritif des Sols

**Hypothèse :** Un bilan nutritif positif du sol (plus d'apports que d'exportations) est associé à de meilleurs rendements à long terme.

In [ ]:
print(f'Colonnes bilan sols : {list(df_sols.columns)}')
print(f'Produits : {df_sols["Produit"].unique()[:10]}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Bilan Nutritif des Sols (kg/ha)', fontsize=15, fontweight='bold')

# Distribution
axes[0].hist(df_sols['Valeur'].dropna(), bins=60, color='saddlebrown', alpha=0.75, edgecolor='white')
axes[0].axvline(0, color='red', lw=2, ls='--', label='Équilibre (0 kg/ha)')
axes[0].set_xlabel('Bilan nutritif (kg/ha)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution du bilan nutritif mondial')
axes[0].legend()

# Évolution temporelle
bns_time = df_sols.groupby('Annee')['Valeur'].median()
axes[1].plot(bns_time.index, bns_time.values, color='saddlebrown', lw=2.5, marker='o', markersize=3)
axes[1].axhline(0, color='red', lw=1.5, ls='--', label='Équilibre')
axes[1].fill_between(bns_time.index, bns_time.values, 0,
                     where=(bns_time.values >= 0), alpha=0.2, color='green', label='Surplus')
axes[1].fill_between(bns_time.index, bns_time.values, 0,
                     where=(bns_time.values < 0), alpha=0.2, color='red', label='Déficit')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Bilan médian (kg/ha)')
axes[1].set_title('Évolution du bilan nutritif médian mondial')
axes[1].legend()

plt.tight_layout()
plt.show()

print('📌 Un bilan négatif persistant = appauvrissement des sols → risque sur rendements futurs.')

---
## 9. Matrice de Corrélation — Vue d'ensemble

**Objectif :** Identifier quelles variables sont les plus corrélées avec le rendement pour orienter le feature engineering.

In [ ]:
# Construction d'un dataset consolidé par (Pays, Annee)
df_full = df_rend_agg.copy()

merges = [
    (df_temp_agg,     'Temperature_C'),
    (df_precip_agg,   'Precipitations_mm'),
    (df_fert_agg,     'Engrais_kgha'),
    (df_pest_agg,     'Pesticides_kgha'),
]

for df_m, col in merges:
    df_full = pd.merge(df_full, df_m[['Pays', 'Annee', col]], on=['Pays', 'Annee'], how='left')

# Ajout de la variable Annee brute
df_full['Annee'] = df_full['Annee'].astype(int)

print(f'Dataset consolidé : {df_full.shape}')
print(f'Taux de remplissage :\n{df_full.notna().mean().round(3)}')
df_full.head()

In [ ]:
corr_cols = ['Rendement', 'Annee', 'Temperature_C', 'Precipitations_mm', 'Engrais_kgha', 'Pesticides_kgha']
corr_matrix = df_full[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    ax=ax,
    linewidths=0.5,
    square=True
)
ax.set_title('Matrice de Corrélation — Variables vs Rendement', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr_avec_rendement = corr_matrix['Rendement'].drop('Rendement').sort_values(key=abs, ascending=False)
print('\n📊 Corrélations avec le Rendement (force absolue décroissante) :')
for var, val in corr_avec_rendement.items():
    sens = '↑ positif' if val > 0 else '↓ négatif'
    print(f'  {var:25s} → {val:+.3f}  ({sens})')

---
## 10. Pairplot — Relations multi-variables

In [ ]:
# Pairplot sur un échantillon pour ne pas saturer la mémoire
df_pair_sample = df_full[corr_cols].dropna().sample(min(3000, len(df_full)), random_state=42)

g = sns.pairplot(
    df_pair_sample,
    vars=['Rendement', 'Temperature_C', 'Precipitations_mm', 'Engrais_kgha'],
    diag_kind='kde',
    plot_kws={'alpha': 0.2, 's': 10, 'color': ACCENT},
    diag_kws={'color': ACCENT, 'fill': True}
)
g.figure.suptitle('Pairplot — Relations entre variables climatiques, intrants et rendement', y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## 11. Disparité géographique & Effet Continent

**Hypothèse :** Les conditions géographiques regroupent des patterns qui ne sont pas capturés par les features individuelles → un encodage régional peut aider le modèle.

In [ ]:
# Top 20 pays sur période récente vs leurs intrants
recents_full = df_full[df_full['Annee'] >= 2010].dropna(subset=['Rendement', 'Engrais_kgha'])

pays_stats = recents_full.groupby('Pays').agg(
    Rendement_med   = ('Rendement',        'median'),
    Engrais_moy     = ('Engrais_kgha',     'mean'),
    Temperature_moy = ('Temperature_C',    'mean'),
    Precip_moy      = ('Precipitations_mm','mean'),
).dropna()

top30 = pays_stats.nlargest(30, 'Rendement_med')

fig, ax = plt.subplots(figsize=(14, 7))
sc = ax.scatter(
    top30['Engrais_moy'],
    top30['Rendement_med'],
    c=top30['Temperature_moy'],
    s=top30['Precip_moy'] / 10,
    cmap='RdYlGn_r',
    alpha=0.85,
    edgecolors='grey',
    linewidth=0.5
)
for pays, row in top30.iterrows():
    ax.annotate(pays, (row['Engrais_moy'], row['Rendement_med']),
                fontsize=7, ha='left', va='bottom', alpha=0.85)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Température moyenne (°C)')
ax.set_xlabel('Usage moyen engrais (kg/ha)')
ax.set_ylabel('Rendement médian (kg/ha)')
ax.set_title('Top 30 pays : Rendement vs Engrais\n(Couleur = Température | Taille = Précipitations)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Les pays les plus productifs combinent : engrais élevés ET températures modérées.')
print('   → Interaction "Engrais × Température" à envisager dans le feature engineering.')

---
## 12. ✅ Synthèse — Hypothèses & Recommandations pour la Modélisation

### Résumé des découvertes

| Facteur | Relation avec le rendement | Recommandation modèle |
|---------|---------------------------|----------------------|
| **Année** | Positive (tendance haussiére) | Feature directe + tendance |
| **Température** | Non-linéaire (optimum 10-20°C) | T + T² (feature quadratique) |
| **Précipitations** | Non-linéaire (optimum 500-1000 mm) | P + P² (feature quadratique) |
| **Engrais (NPK)** | Positive (corrélation forte) | Feature clé — dose totale |
| **Pesticides** | Positive modérée | Feature utile |
| **Bilan nutritif** | Positive (surplus = mieux) | Feature à créer : surplus/déficit |
| **Culture (Produit)** | Très forte variance entre cultures | One-hot encoding obligatoire |
| **Pays** | Très forte variance géographique | Encodage target ou embedding |

### Hypothèses validées ✅
1. Le rendement est **log-normalement distribué** → appliquer `log1p()` avant modélisation.
2. La **relation Température–Rendement est quadratique** (stress thermique au-dessus de 25°C).
3. Les **engrais NPK** sont le prédicteur le plus directement corrélé au rendement.
4. L'**interaction Engrais × Température** capture le fait que les intrants sont plus efficaces dans les zones tempérées.

### Dataset final recommandé pour la modélisation
```
Target  : log1p(Rendement_kgha)
Features: Annee, Produit (encodé), Pays (encodé),
          Temperature_C, Temperature_C²,
          Precipitations_mm, Precipitations_mm²,
          Engrais_NPK_total_kgha,
          Pesticides_total_kgha,
          Bilan_nutritif_sols_kgha
```

### Modèles recommandés
- **Random Forest / XGBoost** : robuste aux non-linéarités et aux interactions.
- **LightGBM** : efficace sur gros volumes.
- **Régression Ridge** : baseline interprétable.